In [ ]:
# repo root + config (walk parents; do not use ../..)
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# CADEC G4 refit (LanguageTool)

CADEC’s original G4 run used the heuristic fallback (`g4_pass ≈ 0.996`).
This notebook re-scores **only G4** with real LanguageTool (`en-US`), using the
same `grammar_score` / `g4_pass` definition as
`RQ1_semantic_entropy_linguistic_predictors.ipynb`, then recomputes
`accepted_final` and rewrites both CADEC perturbation CSVs.

Does **not** regenerate perturbations or re-run G1–G3/G5/G6.

In [ ]:
# Setup
import csv
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = PROJECT_ROOT
INTER_DIR = PROJECT_ROOT / "outputs" / "rq3" / "intermediate"
FULL_CSV = INTER_DIR / "rq3_cadec_validated_perturbations_full.csv"
OUT_PERT = INTER_DIR / "rq3_cadec_perturbations.csv"

# LanguageTool needs a JRE. Prefer standalone Temurin (do NOT conda-install
# openjdk into torch_gpu: that can pull GraalPy and break CPython).
_java_candidates = [
    Path.home() / "data" / "jdk" / "temurin-17" / "bin" / "java",
    Path("/usr/bin/java"),
]
_java = next((p for p in _java_candidates if p.is_file()), None)
if _java is not None:
    _java_home = _java.parent.parent if _java.name == "java" else _java
    os.environ["JAVA_HOME"] = str(_java_home)
    os.environ["PATH"] = str(_java.parent) + os.pathsep + os.environ.get("PATH", "")
    print(f"Using Java: {_java}")
else:
    print("[WARN] No local java found — LanguageTool may fall back to remote API")


def _log(msg: str):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}] {msg}")
    sys.stdout.flush()


assert FULL_CSV.is_file(), f"Missing {FULL_CSV}"
TARGET_COLS = [
    "instance_id",
    "perturbation_type",
    "mention_context",
    "perturbation_text",
    "lexical_change_magnitude",
    "gold_mention",
    "gold_cui",
    "accepted_final",
]
print("Perturbation schema:", TARGET_COLS)
_log("Setup OK.")


## 1) Load LanguageTool (must succeed: no heuristic fallback)

In [ ]:
# LanguageTool en-US (same as RQ1 intent)
import language_tool_python

_lt_tool = language_tool_python.LanguageTool("en-US")
_probe = _lt_tool.check("This are wrong.")
assert len(_probe) >= 1, "LanguageTool returned no matches on a known-bad sentence"
print(f"LanguageTool ready — G4 will NOT use heuristic fallback. lang={_lt_tool.language} probe_matches={len(_probe)}")


## 2) RQ1-identical `grammar_score` + G4 pass rule

In [ ]:
# Copied from RQ1 six-gate cell (LanguageTool branch only)
def grammar_score(text: str) -> float:
    """
    G4 grammaticality score using LanguageTool (pre-registered).
    Returns a score in [0, 1] where 1 = no grammar errors.

    Pre-registered threshold: score >= 0.5 to pass G4.
    (Equivalent to fewer than 1 grammar error per 10 tokens on average.)
    """
    # Refit notebook: LanguageTool MUST be available: do not silently heuristic.
    matches = _lt_tool.check(str(text))
    n_errors = len(matches)
    n_tokens = max(len(str(text).split()), 1)
    score = max(0.0, 1.0 - (n_errors / n_tokens))
    return float(score)


def g4_fields(mention_context: str, pert_text: str):
    """Match RQ1 validate_one G4 block exactly."""
    pert_text = str(pert_text)
    mention_context = str(mention_context)
    g4 = grammar_score(pert_text)
    g4_orig = grammar_score(mention_context)
    g4_delta = g4_orig - g4

    severe_malformed = (
        (len(pert_text.split()) < 2)
        or (sum(ch.isalpha() for ch in pert_text) / max(len(pert_text), 1) < 0.20)
        or (sum(ch in ".,:;!?()[]{}-" for ch in pert_text) / max(len(pert_text), 1) > 0.55)
    )
    # RQ1: g4_pass = (not severe_malformed) and (g4 >= 0.50)
    g4_pass = (not severe_malformed) and (g4 >= 0.50)
    return {
        "g4_grammar_score": g4,
        "g4_original_score": g4_orig,
        "g4_score_delta": g4_delta,
        "g4_pass": bool(g4_pass),
    }


# Quick unit check on known-good / known-bad
_bad = grammar_score("This are wrong.")
_good = grammar_score("This is correct.")
print(f"Probe scores: bad={_bad:.3f} good={_good:.3f}")
assert _bad < _good, "LanguageTool scores do not discriminate bad vs good text"
_log("G4 helpers ready.")

## 3) Load CSV, snapshot before stats, refit G4

In [ ]:
# Load + before snapshot
print(f"Reading: {FULL_CSV}")
df = pd.read_csv(FULL_CSV, low_memory=False)
_log(f"Loaded {len(df):,} rows")
assert len(df) == 55_976, f"Expected 55,976 rows, got {len(df):,}"

GATE_COLS = ["g1_pass", "g2_pass", "g3_pass", "g4_pass", "g5_pass", "g6_pass"]
for c in GATE_COLS + ["accepted_final", "perturbation_text", "mention_context"]:
    assert c in df.columns, f"Missing column {c}"


def _bool_rate(s):
    return float(pd.Series(s).fillna(False).astype(bool).mean())


def snapshot(frame, label):
    print(f"\n===== {label} =====")
    print(f"overall acceptance: {_bool_rate(frame['accepted_final']):.4f} "
          f"({int(frame['accepted_final'].fillna(False).astype(bool).sum()):,} / {len(frame):,})")
    print("per-gate pass rates:")
    for g in GATE_COLS:
        print(f"  {g:10s} {_bool_rate(frame[g]):.4f}")
    print("per-type acceptance:")
    rates = (
        frame.groupby("perturbation_type", as_index=False)
        .agg(n=("accepted_final", "size"), n_accepted=("accepted_final", lambda x: int(pd.Series(x).fillna(False).astype(bool).sum())))
    )
    rates["acceptance_rate"] = rates["n_accepted"] / rates["n"].clip(lower=1)
    print(rates.sort_values("perturbation_type").to_string(index=False))
    sys.stdout.flush()
    return {
        "acceptance": _bool_rate(frame["accepted_final"]),
        "g4_pass": _bool_rate(frame["g4_pass"]),
        "g4_score_mean": float(frame["g4_grammar_score"].astype(float).mean()),
    }


# Keep pre-refit G4 for flip diagnostics (dropped before save)
df["_g4_pass_before"] = df["g4_pass"]
df["_g4_score_before"] = df["g4_grammar_score"]

before = snapshot(df, "BEFORE (heuristic G4)")
assert before["g4_pass"] > 0.99, (
    f"Expected heuristic-era g4_pass ~0.996, got {before['g4_pass']:.4f}"
)

In [ ]:
# Refit G4 over unique texts (cache) then map back
from tqdm.auto import tqdm

# Cache grammar_score by exact string: mention_context + perturbation_text
unique_texts = pd.unique(
    pd.concat(
        [df["perturbation_text"].astype(str), df["mention_context"].astype(str)],
        ignore_index=True,
    )
)
_log(f"Unique texts to score with LanguageTool: {len(unique_texts):,}")

score_cache = {}
for t in tqdm(unique_texts, desc="LanguageTool G4"):
    score_cache[t] = grammar_score(t)

_log(f"Cached {len(score_cache):,} grammar scores")

g4_scores = []
g4_origs = []
g4_deltas = []
g4_passes = []

for mc, pt in zip(df["mention_context"].astype(str), df["perturbation_text"].astype(str)):
    g4 = score_cache[pt]
    g4_orig = score_cache[mc]
    g4_delta = g4_orig - g4
    severe_malformed = (
        (len(pt.split()) < 2)
        or (sum(ch.isalpha() for ch in pt) / max(len(pt), 1) < 0.20)
        or (sum(ch in ".,:;!?()[]{}-" for ch in pt) / max(len(pt), 1) > 0.55)
    )
    g4_pass = (not severe_malformed) and (g4 >= 0.50)
    g4_scores.append(g4)
    g4_origs.append(g4_orig)
    g4_deltas.append(g4_delta)
    g4_passes.append(bool(g4_pass))

df["g4_grammar_score"] = g4_scores
df["g4_original_score"] = g4_origs
df["g4_score_delta"] = g4_deltas
df["g4_pass"] = g4_passes

_log(
    f"G4 refit done | g4_pass={_bool_rate(df['g4_pass']):.4f} "
    f"| mean score={df['g4_grammar_score'].mean():.4f}"
)

## 4) Recompute `accepted_final` (RQ1 conjunction) + after stats

In [ ]:
# accepted_final: RQ1 validate_one conjunction
# RQ1: accepted = all([g1_pass, g2_pass, g3_pass, g4_pass, g5_pass, g6_pass, noop_pass])
# noop_pass encoded as fallback_noop_rejected == 0 when that column exists.

def _as_bool(s):
    return pd.Series(s).fillna(False).astype(bool)


conj = (
    _as_bool(df["g1_pass"])
    & _as_bool(df["g2_pass"])
    & _as_bool(df["g3_pass"])
    & _as_bool(df["g4_pass"])
    & _as_bool(df["g5_pass"])
    & _as_bool(df["g6_pass"])
)
if "fallback_noop_rejected" in df.columns:
    noop_pass = df["fallback_noop_rejected"].fillna(0).astype(int).eq(0)
    df["accepted_final"] = conj & noop_pass
    _log("accepted_final = G1∧G2∧G3∧G4∧G5∧G6 ∧ ¬fallback_noop_rejected (RQ1 exact)")
else:
    df["accepted_final"] = conj
    _log("accepted_final = G1∧G2∧G3∧G4∧G5∧G6")

# Diversity-rescue flags from the original run are stale once G4 changes
if "accepted_relaxed" in df.columns:
    df["accepted_relaxed"] = False

after = snapshot(df, "AFTER (LanguageTool G4)")

print("\n===== DELTA =====")
print(f"acceptance: {before['acceptance']:.4f} → {after['acceptance']:.4f} "
      f"(Δ={after['acceptance'] - before['acceptance']:+.4f})")
print(f"g4_pass:    {before['g4_pass']:.4f} → {after['g4_pass']:.4f} "
      f"(Δ={after['g4_pass'] - before['g4_pass']:+.4f})")
print(f"g4_score μ: {before['g4_score_mean']:.4f} → {after['g4_score_mean']:.4f}")
sys.stdout.flush()

# Assert LanguageTool actually changed G4. Heuristic scores clustered ~0.92 with
# pass≈0.996; real LT can still have a high pass rate (MedMentions ~0.9975), so
# require score-distribution movement and/or pass flips: not pass<0.99.
_n_flip = int((_as_bool(df["g4_pass"]) != _as_bool(df["_g4_pass_before"])).sum())
_score_delta = abs(after["g4_score_mean"] - before["g4_score_mean"])
_pass_delta = abs(after["g4_pass"] - before["g4_pass"])
print(f"G4 pass flips: {_n_flip:,} | score mean Δ={_score_delta:.4f} | pass Δ={_pass_delta:.4f}")
assert _n_flip > 0 or _score_delta > 1e-3 or _pass_delta > 1e-4, (
    "G4 unchanged after LanguageTool refit — still identical to heuristic run."
)
df.drop(columns=["_g4_pass_before", "_g4_score_before"], inplace=True, errors="ignore")
_log("ASSERT OK: G4 refit applied LanguageTool scores (not unchanged heuristic).")

## 5) Save full CSV + BioASQ-schema export

In [ ]:
# Write outputs
print(f"Writing full: {FULL_CSV}")
df.to_csv(
    FULL_CSV,
    index=False,
    quoting=csv.QUOTE_MINIMAL,
    escapechar="\\",
)
_log(f"Wrote {len(df):,} rows → {FULL_CSV}")

missing = [c for c in TARGET_COLS if c not in df.columns]
assert not missing, f"Missing columns: {missing}"
df_export = df[TARGET_COLS].copy()
print(f"Writing CADEC perts: {OUT_PERT}")
df_export.to_csv(
    OUT_PERT,
    index=False,
    quoting=csv.QUOTE_MINIMAL,
    escapechar="\\",
)
_log(f"Wrote {len(df_export):,} rows → {OUT_PERT}")
print("Export columns:", list(df_export.columns))
assert list(df_export.columns) == TARGET_COLS
assert int(df_export["accepted_final"].fillna(False).astype(bool).sum()) > 0
_log("Done.")

# Close LanguageTool server if local
try:
    _lt_tool.close()
except Exception:
    pass
